# CEOS 全量计算 — 可复现运行记录

本笔记本依次调用 `00` ~ `04` 六个脚本，完成全部 CEOS 计算与分析，
并保留打印输出以供核验。

| 步骤 | 脚本 | 说明 |
|------|------|------|
| 0 | `00_prepare_cache.py` | 缓存数据预处理 (星历插值、Kepler 映射) |
| 0b | `00b_find_decay_boundary.py` | CEOS 信号衰减边界探测 |
| 1 | `01_compute_sg.py` | 黑子 CEOS 全量计算 (Algo 1/2/3 + Kuiper + 子集扫描 + 太阳周) |
| 2 | `02_compute_sf.py` | 耀斑 CEOS 全量计算 (同上) |
| 3 | `03_analyze_results.py` | 结果汇总分析 + 可视化图表 |
| 4 | `04_deep_sg_analysis.py` | 黑子分组深度分析 (稀释证据 + 热图) |

**当前参数**: `N_SIM_ALGO12=50000`, `N_SIM_SUBSET=50000`。  
黑子窗口 w=1-5，耀斑窗口 Algo1/2 w=1-30, Algo3 w=1-10，子集扫描和太阳周 w=1-5。

---
## Step 0: 环境与数据检查

In [1]:
import os, glob

# 定位项目根目录 (兼容 VS Code 和 Jupyter Lab 两种 cwd)
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, 'notebooks', '04_asymmetric')):
    PROJECT_ROOT = _cwd
elif os.path.basename(_cwd) == '04_asymmetric':
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, '..', '..'))
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, '..', '..'))

os.chdir(PROJECT_ROOT)
print(f'工作目录: {os.getcwd()}')

SCRIPT_DIR = os.path.join('notebooks', '04_asymmetric')
assert os.path.isfile(os.path.join(SCRIPT_DIR, '01_compute_sg.py')), '找不到脚本!'

SG_CACHE = os.path.join(PROJECT_ROOT, 'results', '04_asymmetric', 'sg', 'cache_data')
SF_CACHE = os.path.join(PROJECT_ROOT, 'results', '04_asymmetric', 'sf', 'cache_data')

print(f'黑子缓存:   {os.path.exists(SG_CACHE)}')
print(f'耀斑缓存:   {os.path.exists(SF_CACHE)}')

for label, cache in [('黑子', SG_CACHE), ('耀斑', SF_CACHE)]:
    pqs = sorted(glob.glob(os.path.join(cache, 'ready_*.parquet')))
    print(f'\n{label} 阶段文件 ({len(pqs)}):') 
    for p in pqs:
        sz = os.path.getsize(p) / 1024
        print(f'  {os.path.basename(p):30s}  {sz:>8.1f} KB')

# 检查缓存是否完整
sg_ok = len(glob.glob(os.path.join(SG_CACHE, 'ready_*.parquet'))) >= 5
sf_ok = len(glob.glob(os.path.join(SF_CACHE, 'ready_*.parquet'))) >= 1
if sg_ok and sf_ok:
    print('\n✅ 缓存完整, 可跳过 Step 0.5, 直接运行 Step 1')
else:
    print('\n⚠️ 缓存不完整, 请运行下方 Step 0.5 生成缓存')

工作目录: /home/bml/data/202603二修版
黑子缓存:   False
耀斑缓存:   False

黑子 阶段文件 (0):

耀斑 阶段文件 (0):

⚠️ 缓存不完整, 请运行下方 Step 0.5 生成缓存


---
## Step 0.5: 缓存生成 (`00_prepare_cache.py`) [可选]

**仅在缓存不完整时需要运行。** 从原始 CSV + 星历 Parquet 生成插值缓存。  
首次运行约 10-15 秒，之后可跳过。

In [2]:
# 如缓存已完整, 可跳过此 cell
%run notebooks/04_asymmetric/00_prepare_cache.py

  CEOS 缓存数据预处理
  输入: /home/bml/data/202603二修版/data/ready
  黑子输出: /home/bml/data/202603二修版/results/04_asymmetric/sg/cache_data
  耀斑输出: /home/bml/data/202603二修版/results/04_asymmetric/sf/cache_data

Step 1: 黑子缓存生成
[星历] 加载: /home/bml/data/202603二修版/data/ready/781_planets_dwarfs_asteroids_lonlat.parquet
  星历天数: 73780, 天体数: 781
  保存: ephem_matrix_8p.npy ((73780, 8))
  保存: kepler_prob_maps.pkl (8 行星)

  处理 All 阶段 (插值)...
    去重: 256859 → 256824
    保存: ready_All.parquet (256824 条)

  处理子阶段 (4 个)...
    保存: ready_Daily.parquet (8301 条)
    保存: ready_Dissipation.parquet (27870 条)
    保存: ready_Duration.parquet (75678 条)
    保存: ready_Onset.parquet (33278 条)

  黑子缓存完成. 耗时: 10.5s

Step 2: 耀斑缓存生成
[星历] 加载: /home/bml/data/202603二修版/data/ready/781_planets_dwarfs_asteroids_lonlat.parquet
  星历天数: 73780, 天体数: 781
  保存: ephem_matrix_8p.npy ((73780, 8))
  保存: kepler_prob_maps.pkl (8 行星)

  加载耀斑: flare_1975_2017.csv
    去重: 39039 → 39034
  插值 39034 条耀斑记录...
    保存: ready_Flare_All.parquet (39034 条)

  耀斑缓存

---
## Step 0.8: 测试黑子和耀斑的衰减边界 (`00b_find_decay_boundary.py`)

In [3]:
%run notebooks/04_asymmetric/00b_find_decay_boundary.py

[algo_workers] ✅ CuPy GPU 加速已启用 (4090)

  Flare Total (SF): w=1-50, N_SIM=10000
  数据量 N=39034, 星历天数 T=73780

    w     窗口    ~天数 | Conj_Ratio      Z        p |  Opp_Ratio      Z        p
  --- ------ ------ | ---------- ------ -------- | ---------- ------ --------
[algo_workers] ✅ CuPy GPU 加速已启用 (4090)
    1     2°   0.1d |    105.7%   +1.9  0.0640   |    102.4%   +0.8  0.4294  
    2     4°   0.3d |    107.3%   +3.0  0.0038** |    100.2%   +0.1  0.9325  
    3     6°   0.5d |    106.3%   +2.9  0.0030** |     99.7%   -0.1  0.8731  
    4     8°   0.6d |    105.1%   +2.5  0.0126*  |    100.0%   +0.0  0.9825  
    5    10°   0.8d |    105.3%   +2.8  0.0038** |    100.9%   +0.5  0.6107  
    6    12°   0.9d |    105.7%   +3.1  0.0014** |    101.1%   +0.6  0.5301  
    7    14°   1.1d |    105.7%   +3.2  0.0002** |    101.0%   +0.6  0.5569  
    8    16°   1.2d |    105.3%   +3.2  0.0010** |    100.8%   +0.5  0.6497  
    9    18°   1.4d |    104.9%   +3.0  0.0024** |    100.6%   +0.4  0.7

---
## Step 1: 黑子 CEOS 全量计算 (`01_compute_sg.py`)

计算内容:
- Algo 1 (Total Pairs): 5 阶段 × 分组 × w=1-5 × 冲合
- Algo 2 (At Least One): 同上
- Algo 3 (单体 781 天体): 5 阶段 × w=1-5 × 冲合 + FDR 校正  
- Kuiper 检验
- 255 子集扫描 (含/不含地球分开保存)
- 太阳周分段 SC12-SC25

In [4]:
%run notebooks/04_asymmetric/01_compute_sg.py

黑子 CEOS 全量计算
模拟次数: Algo1/2=50000, Algo3=100, Subset=50000
窗口: Algo1/2=1-5, Algo3/Subset/Cycle=1-5
并行核心数: 30

找到 5 个阶段文件: ['All', 'Daily', 'Dissipation', 'Duration', 'Onset']

Step 1: Algo 1 (Total Pairs) w=1-5
  使用 7 行星: ['Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune']
  处理阶段: All ...
    分组: Small <100 (N=151201)
    分组: Medium 100-500 (N=86883)
    分组: Large 500-2000 (N=18110)
    分组: XLarge >2000 (N=630)
    分组: Total (N=256824)
  处理阶段: Daily ...
    分组: Small <100 (N=8268)
    分组: Medium 100-500 (N=29)
    分组: Large 500-2000 (N=4)
    分组: Total (N=8301)
  处理阶段: Dissipation ...
    分组: Small <100 (N=27541)
    分组: Medium 100-500 (N=318)
    分组: Large 500-2000 (N=11)
    分组: Total (N=27870)
  处理阶段: Duration ...
    分组: Small <100 (N=69759)
    分组: Medium 100-500 (N=5813)
    分组: Large 500-2000 (N=106)
    分组: Total (N=75678)
  处理阶段: Onset ...
    分组: Small <100 (N=31241)
    分组: Medium 100-500 (N=1984)
    分组: Large 500-2000 (N=53)
    分组: Total (N=33278)
  Algo

---
## Step 2: 耀斑 CEOS 全量计算 (`02_compute_sf.py`)

计算内容:
- Algo 1 (Total Pairs): Flare_All × 分组(B/C/M/X/Total) × w=1-30 × 冲合
- Algo 2 (At Least One): 同上
- Algo 3 (单体 781 天体): w=1-10 × 冲合 + FDR 校正
- Kuiper 检验
- 255 子集扫描 (含/不含地球分开保存)
- 太阳周分段 SC21-SC24

In [5]:
%run notebooks/04_asymmetric/02_compute_sf.py

耀斑 CEOS 全量计算
模拟次数: Algo1/2=50000, Algo3=100, Subset=50000
窗口: Algo1/2=1-30, Algo3=1-10, Subset/Cycle=1-5
并行核心数: 30

找到 1 个阶段文件: ['Flare_All']

Step 1: Algo 1 (Total Pairs) w=1-30
  使用 7 行星: ['Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune']
  处理阶段: Flare_All ...
    分组: B-Class (N=7127)
    分组: C-Class (N=26654)
    分组: M-Class (N=4824)
    分组: X-Class (N=429)
    分组: Total (N=39034)
  Algo 1 完成. 耗时: 126.2s
  输出文件: /home/bml/data/202603二修版/results/04_asymmetric/sf/sf_algo1_total_pairs.csv

Step 2: Algo 2 (At Least One) w=1-30
  使用 7 行星: ['Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune']
  处理阶段: Flare_All ...
    分组: B-Class (N=7127)
    分组: C-Class (N=26654)
    分组: M-Class (N=4824)
    分组: X-Class (N=429)
    分组: Total (N=39034)
  Algo 2 完成. 耗时: 132.1s
  输出文件: /home/bml/data/202603二修版/results/04_asymmetric/sf/sf_algo2_at_least_one.csv

Step 3: Algo 3 (单体 781 天体) w=1-10
  处理阶段: Flare_All ...
    分析 781 天体 vs 39034 条记录...
    处理 781 天体完成.
  Algo 3

---
## Step 3: 结果汇总分析 + 可视化 (`03_analyze_results.py`)

产出:
- Algo 1/2 衰减汇总表 (w=1-20)
- 各分组 Conj vs Opp 对比
- 子集扫描不对称分布
- 太阳周分段一致性
- Kuiper 检验排名
- 图表保存至 `results/03_ceos/figures/`

In [6]:
%run notebooks/04_asymmetric/03_analyze_results.py

  CEOS 计算结果分析报告

  Flare — Algo 1 (Total Pairs)

  【Flare_All】 Total:
  Window  Conj_Ratio    Conj_p   Conj_Effect  │   Opp_Ratio     Opp_p    Opp_Effect
  ──────  ──────────  ────────  ────────────  │  ──────────  ────────  ────────────
       1     105.7%    0.0665    Enhancement  │     102.5%    0.4235    Enhancement
       2     107.3%    0.0033**   Enhancement  │     100.2%    0.9233    Enhancement
       3     106.3%    0.0036**   Enhancement  │      99.7%    0.8705    Suppression
       5     105.3%    0.0044**   Enhancement  │     100.9%    0.6153    Enhancement
      10     105.0%    0.0010***   Enhancement  │     101.1%    0.4830    Enhancement
      15     103.5%    0.0147*   Enhancement  │     100.2%    0.8973    Enhancement
      20     102.5%    0.0459*   Enhancement  │      99.7%    0.7973    Suppression

  📊 图表已保存: /home/bml/data/202603二修版/results/04_asymmetric/analysis/flare_algo_1_(total_pairs)_decay.png

  Flare — Algo 2 (At Least One)

  【Flare_All】 Total:
  Window 

---
## Step 4: 黑子分组深度分析 (`04_deep_sg_analysis.py`)

产出:
- 面积梯度 Ratio 表 (稀释证据)
- 生命周期阶段 × 面积交叉分析
- 不对称信号强度对比 (耀斑 vs 黑子)
- 图表: 分组 Asym 曲线、衰减对比、阶段×面积热图

In [7]:
%run notebooks/04_asymmetric/04_deep_sg_analysis.py



  1. 黑子面积分组 Ratio 梯度 (稀释证据)

  Algo 1 - All Stage - Conjunction Ratio by Area:
                 Group        N       w=1       w=2       w=3       w=5
  ────────────────────  ───────  ────────  ────────  ────────  ────────
            Small <100   151201    100.7%     99.7%     99.9%     99.5%
        Medium 100-500    86883     98.6%     99.9%    100.1%    100.3%
        Large 500-2000    18110    102.2%    104.3%    101.5%    100.5%
          XLarge >2000      630     93.8%    104.0%    111.5%    109.3%
                 Total   256824    100.1%    100.1%    100.1%     99.9%

  Opposition Ratio by Area:
                 Group        N       w=1       w=2       w=3       w=5
  ────────────────────  ───────  ────────  ────────  ────────  ────────
            Small <100   151201    100.0%    101.1%    101.0%    100.8%
        Medium 100-500    86883    101.6%    101.8%    100.7%    100.0%
        Large 500-2000    18110     99.4%    104.2%    102.1%    100.9%
          XLarge >2000    

---
## Step 5: 输出文件清单

In [8]:
import os

base = os.path.join(os.getcwd(), 'results', '04_asymmetric')
total_size = 0
n_files = 0

print('=' * 65)
print('  results/04_asymmetric/ 输出文件清单')
print('=' * 65)

for sub in ['sg', 'sf', 'figures']:
    d = os.path.join(base, sub)
    if not os.path.isdir(d):
        continue
    print(f'\n  [{sub}/]')
    for f in sorted(os.listdir(d)):
        fp = os.path.join(d, f)
        if os.path.isdir(fp):
            continue
        if not f.endswith(('.csv', '.png')):
            continue
        sz = os.path.getsize(fp)
        total_size += sz
        n_files += 1
        unit = 'MB' if sz > 1024*1024 else 'KB'
        val = sz/1024/1024 if sz > 1024*1024 else sz/1024
        print(f'    {f:45s}  {val:>8.1f} {unit}')

print(f'\n  共 {n_files} 个文件, 总计 {total_size/1024/1024:.1f} MB')

  results/04_asymmetric/ 输出文件清单

  [sg/]
    sg_Large_500-2000_solar_cycle_segment.csv          69.9 KB
    sg_Large_500-2000_subset_scan_no_earth.csv         61.1 KB
    sg_Large_500-2000_subset_scan_with_earth.csv       64.0 KB
    sg_SC12_subset_scan_no_earth.csv                   35.6 KB
    sg_SC12_subset_scan_with_earth.csv                 37.4 KB
    sg_SC13_subset_scan_no_earth.csv                   36.1 KB
    sg_SC13_subset_scan_with_earth.csv                 37.8 KB
    sg_SC14_subset_scan_no_earth.csv                   35.6 KB
    sg_SC14_subset_scan_with_earth.csv                 37.4 KB
    sg_SC15_subset_scan_no_earth.csv                   36.2 KB
    sg_SC15_subset_scan_with_earth.csv                 37.9 KB
    sg_SC16_subset_scan_no_earth.csv                   36.0 KB
    sg_SC16_subset_scan_with_earth.csv                 37.6 KB
    sg_SC17_subset_scan_no_earth.csv                   36.0 KB
    sg_SC17_subset_scan_with_earth.csv                 38.1 KB
    sg_SC18_su